# 04 – Thermal Tiles Inventory

**Objective:** Catalogue known thermal tile resources from the Gdynia viewer without performing a
full bulk download.  The output is a tile inventory CSV at
`outputs/tables/thermal_tiles_inventory.csv`.

> **Note:** Full tile download is deferred to Phase 2.  See
> [docs/phase2_outlook.md](../docs/phase2_outlook.md) and
> [docs/limitations.md](../docs/limitations.md).

In [ ]:
from __future__ import annotations

import mercantile
import pandas as pd
import requests
import yaml
from pathlib import Path

In [ ]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
PROJECT_ROOT  = Path.cwd().parent
CONFIG_PATH   = PROJECT_ROOT / "config" / "config.yaml"

with CONFIG_PATH.open() as fh:
    cfg = yaml.safe_load(fh)

VIEWER_BASE   = cfg["sources"]["viewer"]["base_url"]
INVENTORY_OUT = PROJECT_ROOT / cfg["database"]["thermal_inventory_path"]
INVENTORY_OUT.parent.mkdir(parents=True, exist_ok=True)

# Approximate bounding box for Gdynia (WGS 84)
GDYNIA_BBOX = (18.43, 54.45, 18.61, 54.57)  # (west, south, east, north)

# Zoom levels to catalogue (low-resolution overview only)
ZOOM_LEVELS = [10, 12, 14]

print("Viewer base URL:", VIEWER_BASE)

## 1. Enumerate tiles for the Gdynia bounding box

In [ ]:
west, south, east, north = GDYNIA_BBOX
rows = []

for zoom in ZOOM_LEVELS:
    tiles = list(mercantile.tiles(west, south, east, north, zooms=zoom))
    for t in tiles:
        bounds = mercantile.bounds(t)
        rows.append({
            "zoom": t.z,
            "x": t.x,
            "y": t.y,
            "west": bounds.west,
            "south": bounds.south,
            "east": bounds.east,
            "north": bounds.north,
            "confirmed": False,
        })

inventory = pd.DataFrame(rows)
print(f"Enumerated {len(inventory):,} tiles across zoom levels {ZOOM_LEVELS}")
inventory.groupby("zoom").size().rename("tile_count")

## 2. Probe a small sample to verify tile URL pattern

We probe only a handful of tiles to confirm the URL structure – **not** a bulk download.

In [ ]:
# Placeholder tile URL template – update once the actual pattern is confirmed
# from conf.js / viewer inspection.
TILE_URL_TEMPLATE = VIEWER_BASE + "tiles/{z}/{x}/{y}.png"

SAMPLE_SIZE = 3
sample = inventory[inventory["zoom"] == ZOOM_LEVELS[0]].head(SAMPLE_SIZE).copy()

def probe_tile(row: pd.Series) -> int | None:
    url = TILE_URL_TEMPLATE.format(z=row["zoom"], x=row["x"], y=row["y"])
    try:
        r = requests.head(url, timeout=10, allow_redirects=True)
        return r.status_code
    except requests.RequestException:
        return None

sample["http_status"] = sample.apply(probe_tile, axis=1)
print("Sample probe results:")
sample[["zoom", "x", "y", "http_status"]]

## 3. Save inventory

In [ ]:
inventory.to_csv(INVENTORY_OUT, index=False)
print(f"Saved {len(inventory):,} rows to {INVENTORY_OUT}")